## Definitions

$\text{Accuracy} =
\frac{\text{correct classifications}}{\text{total classifications}}
= \frac{TP+TN}{TP+TN+FP+FN} $

$\text{Recall (or TPR)} =
\frac{\text{correctly classified actual positives}}{\text{all actual positives}}
= \frac{TP}{TP+FN} $

$\text{FPR} =
\frac{\text{incorrectly classified actual negatives}}
{\text{all actual negatives}}
= \frac{FP}{FP+TN}$

$\text{Precision} =
\frac{\text{correctly classified actual positives}}
{\text{everything classified as positive}}
= \frac{TP}{TP+FP} $

## Decision Tree

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, roc_auc_score
import os

path = os.getenv("DATA_PATH")
df = pd.read_csv(path)
target = "Problem_SKU"
seed = 1337 
# One-hot encode Storage_Size (drop size_4 as baseline)
size_dummies = pd.get_dummies(df['Storage_Size'], prefix='Size', drop_first=True)

# Encode Defect_In_Linked_Receive as 0/1
defect_linked_num = df['Defect_In_Linked_Receive'].astype(int)

# Numeric features (keep standardized)
numeric_features = [
    "Global_SKU_Defect_Rate_%_std",
    "ABS_Volume_Difference_std",
    "Aisle_Hold_%_std",
    "#_Pick_Events_std",
    "#_Pick_Events_In_Clique_std",
    "#_Picks_std",
    "#_Picks_In_Clique_std",
    "Time_In_Loc_std",
    "Current_Max_Volume_std",
]

feature_cols = numeric_features + list(size_dummies.columns) + ['Defect_In_Linked_Receive']

# Combine all properly encoded features
X = df[numeric_features].copy()
X = pd.concat([X, size_dummies, defect_linked_num], axis=1)
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=seed, stratify=y
)

dt = DecisionTreeClassifier(
    max_depth=None,        # let it grow; you can cap later
    min_samples_leaf=50,   # mild regularization
    random_state=seed,
)

dt.fit(X_train, y_train)

y_pred_dt = dt.predict(X_test)
y_proba_dt = dt.predict_proba(X_test)[:, 1]

print("Decision Tree")
print(classification_report(y_test, y_pred_dt, digits=3))
print("ROC AUC:", roc_auc_score(y_test, y_proba_dt))

import numpy as np

fi_dt = pd.Series(dt.feature_importances_, index=feature_cols).sort_values(ascending=False)
print(fi_dt)

Decision Tree
              precision    recall  f1-score   support

       False      0.945     0.987     0.965     50954
        True      0.621     0.270     0.377      4046

    accuracy                          0.934     55000
   macro avg      0.783     0.629     0.671     55000
weighted avg      0.921     0.934     0.922     55000

ROC AUC: 0.8144272748038606
#_Picks_In_Clique_std           0.308731
Aisle_Hold_%_std                0.146597
Current_Max_Volume_std          0.126887
#_Picks_std                     0.112853
ABS_Volume_Difference_std       0.101935
#_Pick_Events_In_Clique_std     0.091101
Global_SKU_Defect_Rate_%_std    0.064291
#_Pick_Events_std               0.017249
Time_In_Loc_std                 0.013118
Defect_In_Linked_Receive        0.008358
Size_size_3                     0.004815
Size_size_2                     0.003028
Size_size_4                     0.001036
dtype: float64


In [3]:
from sklearn.tree import DecisionTreeClassifier

dt_recall = DecisionTreeClassifier(
    class_weight='balanced',  # ← weights minority class higher
    min_samples_leaf=20,
    random_state=seed
)

dt_recall.fit(X_train, y_train)

y_pred_dt = dt_recall.predict(X_test)
y_proba_dt = dt_recall.predict_proba(X_test)[:, 1]

print("Decision Tree")
print(classification_report(y_test, y_pred_dt, digits=3))
print("ROC AUC:", roc_auc_score(y_test, y_proba_dt))

import numpy as np

fi_dt_recall = pd.Series(dt_recall.feature_importances_, index=feature_cols).sort_values(ascending=False)
print(fi_dt_recall)

Decision Tree
              precision    recall  f1-score   support

       False      0.967     0.793     0.872     50954
        True      0.202     0.661     0.310      4046

    accuracy                          0.783     55000
   macro avg      0.585     0.727     0.591     55000
weighted avg      0.911     0.783     0.830     55000

ROC AUC: 0.772569274437504
#_Picks_In_Clique_std           0.320541
Aisle_Hold_%_std                0.142590
Current_Max_Volume_std          0.109352
#_Picks_std                     0.096826
ABS_Volume_Difference_std       0.080774
#_Pick_Events_In_Clique_std     0.070920
Global_SKU_Defect_Rate_%_std    0.067365
Time_In_Loc_std                 0.044706
#_Pick_Events_std               0.042699
Size_size_3                     0.008345
Size_size_2                     0.007431
Defect_In_Linked_Receive        0.006258
Size_size_4                     0.002191
dtype: float64


## XG Boost

In [4]:
from xgboost import XGBClassifier

xgb = XGBClassifier(
    objective="binary:logistic",
    n_estimators=200,
    max_depth=4,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    n_jobs=-1,
    random_state=seed,
    eval_metric="logloss",
)

xgb.fit(X_train, y_train)

y_pred_xgb = xgb.predict(X_test)
y_proba_xgb = xgb.predict_proba(X_test)[:, 1]

print("XGBoost")
print(classification_report(y_test, y_pred_xgb, digits=3))
print("ROC AUC:", roc_auc_score(y_test, y_proba_xgb))

fi_xgb = pd.Series(xgb.feature_importances_, index=feature_cols).sort_values(ascending=False)
print(fi_xgb)

results = pd.DataFrame({
    "model": ["DecisionTree", "XGBoost"],
    "roc_auc": [roc_auc_score(y_test, y_proba_dt),
                roc_auc_score(y_test, y_proba_xgb)],
})
print(results)

XGBoost
              precision    recall  f1-score   support

       False      0.946     0.990     0.967     50954
        True      0.696     0.284     0.403      4046

    accuracy                          0.938     55000
   macro avg      0.821     0.637     0.685     55000
weighted avg      0.927     0.938     0.926     55000

ROC AUC: 0.8745512342255684
#_Picks_In_Clique_std           0.303107
Aisle_Hold_%_std                0.112710
Current_Max_Volume_std          0.098330
#_Picks_std                     0.083329
Defect_In_Linked_Receive        0.064068
ABS_Volume_Difference_std       0.060908
Size_size_2                     0.059627
#_Pick_Events_In_Clique_std     0.055876
Global_SKU_Defect_Rate_%_std    0.046454
Size_size_3                     0.045545
Size_size_4                     0.045091
#_Pick_Events_std               0.013870
Time_In_Loc_std                 0.011085
dtype: float32
          model   roc_auc
0  DecisionTree  0.772569
1       XGBoost  0.874551


In [5]:
from xgboost import XGBClassifier

xgb_recall = XGBClassifier(
    scale_pos_weight=len(y_train[y_train==0])/len(y_train[y_train==1]),  # ← imbalance ratio
    max_depth=6,
    n_estimators=300
)

xgb_recall.fit(X_train, y_train)

y_pred_xgb = xgb_recall.predict(X_test)
y_proba_xgb = xgb_recall.predict_proba(X_test)[:, 1]

print("XGBoost")
print(classification_report(y_test, y_pred_xgb, digits=3))
print("ROC AUC:", roc_auc_score(y_test, y_proba_xgb))

fi_xgb = pd.Series(xgb_recall.feature_importances_, index=feature_cols).sort_values(ascending=False)
print(fi_xgb)



XGBoost
              precision    recall  f1-score   support

       False      0.968     0.889     0.927     50954
        True      0.312     0.633     0.418      4046

    accuracy                          0.870     55000
   macro avg      0.640     0.761     0.672     55000
weighted avg      0.920     0.870     0.890     55000

ROC AUC: 0.8458725219306003
#_Picks_In_Clique_std           0.171069
Defect_In_Linked_Receive        0.129783
Size_size_4                     0.090412
Size_size_2                     0.085010
Aisle_Hold_%_std                0.084604
Size_size_3                     0.080949
Current_Max_Volume_std          0.067610
#_Picks_std                     0.064731
ABS_Volume_Difference_std       0.054344
#_Pick_Events_In_Clique_std     0.054027
Global_SKU_Defect_Rate_%_std    0.048493
#_Pick_Events_std               0.034562
Time_In_Loc_std                 0.034407
dtype: float32


## SMOTE

In [6]:
import pandas as pd
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, roc_auc_score, precision_recall_curve
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE

print("\n🔄 Applying SMOTE...")
smote = SMOTE(random_state=42, k_neighbors=5)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)
print(f"SMOTE balanced: {y_train_smote.mean():.3%}")

# ========== 3. DECISION TREE (MAX RECALL) ==========
print("\n🌳 Training Decision Tree...")
dt = DecisionTreeClassifier(
    class_weight='balanced',
    min_samples_leaf=10,    # slight smoothing
    max_depth=10,
    random_state=42
)
dt.fit(X_train_smote, y_train_smote)

y_pred_dt = dt.predict(X_test)
y_proba_dt = dt.predict_proba(X_test)[:, 1]

print("\n📊 DECISION TREE RESULTS")
print(classification_report(y_test, y_pred_dt, digits=3, zero_division=0))
print(f"ROC AUC: {roc_auc_score(y_test, y_proba_dt):.3f}")

# ========== 4. XGBOOST (MAX RECALL) ==========
print("\n🚀 Training XGBoost...")
scale_pos_weight = 1.0  # SMOTE already balanced training data
xgb = XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
    eval_metric='logloss'
)
xgb.fit(X_train_smote, y_train_smote)

y_pred_xgb = xgb.predict(X_test)
y_proba_xgb = xgb.predict_proba(X_test)[:, 1]

print("\n📊 XGBOOST RESULTS")
print(classification_report(y_test, y_pred_xgb, digits=3, zero_division=0))
print(f"ROC AUC: {roc_auc_score(y_test, y_proba_xgb):.3f}")

# ========== 5. THRESHOLD OPTIMIZATION (RECALL BOOSTER) ==========
def optimize_recall_threshold(y_true, y_proba, min_precision=0.1):
    """Threshold for max recall above min_precision"""
    precision, recall, thresholds = precision_recall_curve(y_true, y_proba)
    valid = (precision >= min_precision) & (recall > 0)
    
    if valid.sum() == 0:
        best_idx = np.argmax(recall)
    else:
        best_idx = np.argmax(recall[valid])
    
    return thresholds[best_idx], recall[best_idx], precision[best_idx]

thresh_dt, rec_dt, prec_dt = optimize_recall_threshold(y_test, y_proba_dt)
thresh_xgb, rec_xgb, prec_xgb = optimize_recall_threshold(y_test, y_proba_xgb)

print("\n🎯 OPTIMIZED THRESHOLDS")
print(f"DT: thresh={thresh_dt:.3f} → recall={rec_dt:.1%} prec={prec_dt:.1%}")
print(f"XGB: thresh={thresh_xgb:.3f} → recall={rec_xgb:.1%} prec={prec_xgb:.1%}")

# Apply thresholds
y_pred_dt_opt = (y_proba_dt >= thresh_dt).astype(int)
y_pred_xgb_opt = (y_proba_xgb >= thresh_xgb).astype(int)

print("\n📊 DT OPTIMIZED")
print(classification_report(y_test, y_pred_dt_opt, digits=3, zero_division=0))
print("\n📊 XGB OPTIMIZED") 
print(classification_report(y_test, y_pred_xgb_opt, digits=3, zero_division=0))

print("\n✅ MAX RECALL BASELINES COMPLETE")
print("Feature importances:")
print("\nDT Top 5:")
print(pd.Series(dt.feature_importances_, index=X.columns).nlargest(5))
print("\nXGB Top 5:")
print(pd.Series(xgb.feature_importances_, index=X.columns).nlargest(5))


🔄 Applying SMOTE...
SMOTE balanced: 50.000%

🌳 Training Decision Tree...

📊 DECISION TREE RESULTS
              precision    recall  f1-score   support

       False      0.972     0.824     0.892     50954
        True      0.240     0.696     0.356      4046

    accuracy                          0.815     55000
   macro avg      0.606     0.760     0.624     55000
weighted avg      0.918     0.815     0.853     55000

ROC AUC: 0.823

🚀 Training XGBoost...

📊 XGBOOST RESULTS
              precision    recall  f1-score   support

       False      0.960     0.945     0.953     50954
        True      0.425     0.510     0.464      4046

    accuracy                          0.913     55000
   macro avg      0.693     0.728     0.708     55000
weighted avg      0.921     0.913     0.917     55000

ROC AUC: 0.854

🎯 OPTIMIZED THRESHOLDS
DT: thresh=0.000 → recall=100.0% prec=7.4%
XGB: thresh=0.002 → recall=100.0% prec=7.4%

📊 DT OPTIMIZED
              precision    recall  f1-score   su